In [ ]:
from pyGmsh import pyGmsh
from pyGmsh import Numberer, NumberedMesh
import numpy as np
import openseespy.opensees as ops

# Thick-Walled Cylinder — Quarter Model (Lamé Problem)

**Architecture: pyGmsh as mesh provider → OpenSeesPy as external solver**

This example demonstrates a clean separation of concerns:

- **pyGmsh** — geometry, meshing, physical groups, and post-processing views.
  It never imports or calls any solver.
- **OpenSeesPy** — FEM model building, analysis, and result extraction.
  It receives plain numpy arrays from pyGmsh and works independently.
- **Results go back to Gmsh** — we pass computed arrays to `g.view` for
  visualization in the Gmsh GUI. Again, no solver dependency in the View class.

This pattern makes pyGmsh reusable with *any* solver (OpenSees, FEniCS, Abaqus input decks, etc.).

In [ ]:
# ============================================================
# Parameters
# ============================================================
#
# Thick-walled cylinder under internal pressure (Lamé problem):
#   - Quarter model exploiting double symmetry (x-axis and y-axis)
#   - Plane strain assumption (thk = 1.0, out-of-plane strain = 0)
#   - Analytical solution exists → good benchmark for FEM verification
#
inner_radius = 100.0    # [mm]  inner bore radius
outer_radius = 200.0    # [mm]  outer wall radius
lc           = 10.0     # [mm]  characteristic mesh size (controls density)

E   = 210.0e3   # [MPa]  Young's modulus  (steel ≈ 210 GPa)
nu  = 0.3       #  [-]   Poisson's ratio
p   = 100.0     # [MPa]  internal pressure applied on inner arc
thk = 1.0       # [mm]   unit thickness (plane strain formulation)

## Part 1 — Geometry & Mesh (pyGmsh)

Everything here is pure pyGmsh. No solver imports needed.

**Geometry workflow:**
1. Create points with `g.model.add_point()` — the `lc` parameter sets the
   target mesh element size at each point.
2. Connect points with lines and arcs — `add_arc(start, center, end)` uses
   the OCC kernel (not geo), so the center point is required but does **not**
   become a mesh node on the arc.
3. Form a closed curve loop → plane surface.
4. Define physical groups to tag boundaries and the domain — these are how
   we later query which nodes belong to which BC or load.

In [ ]:
# Initialize pyGmsh — the context manager handles gmsh.initialize/finalize,
# but here we use manual .initialize() / .finalize() for notebook flexibility.
g = pyGmsh(model_name="Plate2D", verbose=True)
g.initialize()

# --- Geometry: quarter annulus (OCC kernel) ---
#
#   p3 (0, R_o)
#    |  \  outer arc (l2)
#    |    \
#   l3     p2 (R_o, 0)
#    |    /
#    |  /  inner arc (l4)
#   p4 (0, R_i)          l1
#    ·----- p1 (R_i, 0) ---→ p2 (R_o, 0)
#   pc (0,0)
#
# Note: pc is the arc center. OCC creates it as a geometry point but it
# will NOT lie on the arcs. It does create an orphan mesh node at (0,0)
# which we filter out later via used_tags.

pc = g.model.add_point(0, 0, 0, lc=lc, label="center")
p1 = g.model.add_point(inner_radius, 0, 0, lc=lc, label="inner_x")
p2 = g.model.add_point(outer_radius, 0, 0, lc=lc, label="outer_x")
p3 = g.model.add_point(0, outer_radius, 0, lc=lc, label="outer_y")
p4 = g.model.add_point(0, inner_radius, 0, lc=lc, label="inner_y")

# 2 radial lines + 2 circular arcs (CCW winding)
l1 = g.model.add_line(p1, p2, label="bottom")         # radial line along x-axis
l2 = g.model.add_arc(p2, pc, p3, label="outer_arc")   # outer arc: R_o
l3 = g.model.add_line(p3, p4, label="left")            # radial line along y-axis
l4 = g.model.add_arc(p4, pc, p1, label="inner_arc")   # inner arc: R_i

# Close the loop and create the surface
loop = g.model.add_curve_loop([l1, l2, l3, l4])
surf = g.model.add_plane_surface(loop, label="plate")

# Inspect what we built
g.model.registry()

In [ ]:
# --- Physical groups ---
#
# Physical groups label geometry entities so we can later query
# "give me all mesh nodes on this boundary". The tag returned by
# g.physical.add() is what we pass to g.physical.get_nodes().
#
# dim=1 → curves (boundaries), dim=2 → surfaces (domain)

pg_symY = g.physical.add(1, [l1], name="Sym_Y")      # bottom edge → uy = 0
pg_symX = g.physical.add(1, [l3], name="Sym_X")      # left edge   → ux = 0
pg_pres = g.physical.add(1, [l4], name="Pressure")   # inner arc   → pressure load
pg_plat = g.physical.add(2, [surf], name="Plate")    # domain      → element region

g.physical.summary()

In [ ]:
# --- Generate mesh ---
#
# set_order(1) → linear elements (3-node triangles for 2D)
# generate(2)  → mesh the 2D surfaces
#
# The mesh is stored inside Gmsh's internal state. We extract it
# in the next cell with get_fem_data().

g.mesh.set_order(1)
g.mesh.generate(2)

## Part 2 — Extract mesh data & Renumber

Two paths for mesh extraction:

1. **`get_fem_data()`** — raw Gmsh tags (non-contiguous, may have orphans)
2. **`get_numbered_mesh()`** — contiguous solver-ready IDs via the Numberer

We use both here: `get_fem_data()` for the raw data and boundary queries,
then `get_numbered_mesh()` for the solver. The `NumberedMesh` object provides:

| Attribute               | Description                                    |
|-------------------------|------------------------------------------------|
| `node_ids`              | Contiguous 1-based solver node IDs             |
| `node_coords`           | Coordinates aligned with `node_ids`            |
| `elem_ids`              | Contiguous 1-based solver element IDs          |
| `connectivity`          | Element→node matrix in solver IDs              |
| `gmsh_to_solver_node`   | Gmsh tag → solver ID map                       |
| `solver_to_gmsh_node`   | Solver ID → Gmsh tag map                       |
| `gmsh_to_solver_elem`   | Gmsh element tag → solver element ID           |
| `bandwidth`             | Semi-bandwidth of the connectivity             |

The Numberer automatically handles orphan filtering (`used_only=True`
by default), so the center point at (0,0) is excluded — no manual
`used_tags` check needed.

In [ ]:
# --- Raw FEM data (still needed for boundary queries + plotting) ---
fem = g.mesh.get_fem_data(dim=2)

node_tags    = fem['node_tags']       # (N,)        Gmsh node IDs
node_coords  = fem['node_coords']     # (N, 3)      XYZ
tag_to_idx   = fem['tag_to_idx']      # {gtag: i}   into node_coords
connectivity = fem['connectivity']    # (nElem, 3)   node-tag matrix (Gmsh tags)
elem_tags    = fem['elem_tags']       # [int, ...]   Gmsh element tags
used_tags    = fem['used_tags']       # set[int]     orphan-free node set

# --- Numbered mesh (solver-ready) ---
mesh = g.mesh.get_numbered_mesh(dim=2, method="simple")
print(mesh.summary())

# --- Boundary nodes from physical groups ---
# These come back as Gmsh tags — we translate them to solver IDs
# via the bidirectional map.

bottom_nodes_gmsh = g.physical.get_nodes(1, pg_symY)['tags']   # uy = 0
left_nodes_gmsh   = g.physical.get_nodes(1, pg_symX)['tags']   # ux = 0
inner_nodes_gmsh  = g.physical.get_nodes(1, pg_pres)['tags']   # pressure

# --- Inner arc edges (1D line elements on l4) ---
# For pressure loads, we need the 1D edge connectivity on the inner arc.
# These are in Gmsh tags — we'll translate when applying loads.

inner_elems = g.mesh.get_elements(dim=1, tag=l4)
inner_edges = []
for etype, enodes in zip(inner_elems['types'], inner_elems['node_tags']):
    props = g.mesh.get_element_properties(etype)
    inner_edges = enodes.reshape(-1, props['n_nodes']).astype(int)

print(f"\nNodes: {len(node_tags)} total, {mesh.n_nodes} used (solver)")
print(f"Triangles: {mesh.n_elems}")
print(f"Bandwidth: {mesh.bandwidth}")
print(f"Bottom (uy=0): {len(bottom_nodes_gmsh)} | Left (ux=0): {len(left_nodes_gmsh)}")
print(f"Inner arc: {len(inner_nodes_gmsh)} nodes, {len(inner_edges)} edges")

## Part 3 — OpenSees Model (using Numberer)

The `NumberedMesh` gives us contiguous 1-based IDs and connectivity
ready for OpenSees. No more manual `gmsh_to_ops` loop — the Numberer
handles orphan filtering, contiguous renumbering, and bidirectional
maps automatically.

For boundary conditions and loads, we use `mesh.gmsh_to_solver_node`
to translate Gmsh physical-group tags to solver IDs.

In [ ]:
ops.wipe()
ops.model("basic", "-ndm", 2, "-ndf", 2)

# --- Nodes (from NumberedMesh) ---
# Contiguous IDs, orphans already filtered out.
for i in range(mesh.n_nodes):
    nid = int(mesh.node_ids[i])
    x, y = float(mesh.node_coords[i, 0]), float(mesh.node_coords[i, 1])
    ops.node(nid, x, y)

# --- Material ---
ops.nDMaterial("ElasticIsotropic", 1, E, nu)

# --- Elements (from NumberedMesh) ---
# connectivity is already in solver IDs — feed directly to OpenSees.
for i in range(mesh.n_elems):
    eid = int(mesh.element_ids[i])
    n1, n2, n3 = [int(n) for n in mesh.connectivity[i]]
    ops.element("tri31", eid, n1, n2, n3, thk, "PlaneStrain", 1)

# --- Boundary conditions (symmetry) ---
# Translate Gmsh physical-group tags → solver IDs via the map.
for gtag in bottom_nodes_gmsh.astype(int):
    sid = mesh.gmsh_to_solver_node.get(int(gtag))
    if sid is not None:
        ops.fix(sid, 0, 1)   # fix uy, free ux

for gtag in left_nodes_gmsh.astype(int):
    sid = mesh.gmsh_to_solver_node.get(int(gtag))
    if sid is not None:
        ops.fix(sid, 1, 0)   # fix ux, free uy

print(f"OpenSees model: {mesh.n_nodes} nodes, {mesh.n_elems} elements")

In [ ]:
# --- Consistent nodal forces for internal pressure ---
#
# We cannot apply a distributed pressure directly in OpenSees tri31.
# Instead we compute equivalent nodal forces by integrating the traction
# over each edge element on the inner arc.
#
# For a 2-node linear edge of length L with traction t_i, t_j at each node:
#
#     F_i = (L/6)(2·t_i + t_j)
#     F_j = (L/6)(t_i + 2·t_j)
#
# This comes from the consistent load vector: F = ∫ N^T · t dΓ
# using linear shape functions N = [(1-ξ)/2, (1+ξ)/2].
#
# The traction direction is radially outward: t = p · n̂,  n̂ = (x/r, y/r)
#
# Note: inner_edges contains Gmsh node tags. We look up coordinates
# via tag_to_idx (raw Gmsh arrays) and solver IDs via the Numberer map.

ops.timeSeries("Linear", 1)
ops.pattern("Plain", 1, 1)

nodal_forces = {}

for edge in inner_edges:
    n1g, n2g = int(edge[0]), int(edge[1])
    i1, i2 = tag_to_idx[n1g], tag_to_idx[n2g]

    # Coordinates of edge endpoints
    x1, y1 = node_coords[i1, 0], node_coords[i1, 1]
    x2, y2 = node_coords[i2, 0], node_coords[i2, 1]

    # Edge length and radii
    Le = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    r1 = np.sqrt(x1**2 + y1**2)
    r2 = np.sqrt(x2**2 + y2**2)

    # Traction at each node: t = p · (x/r, y/r)
    tx1, ty1 = p * x1/r1, p * y1/r1
    tx2, ty2 = p * x2/r2, p * y2/r2

    # Consistent nodal forces
    Fx1 = (Le/6) * (2*tx1 + tx2);  Fy1 = (Le/6) * (2*ty1 + ty2)
    Fx2 = (Le/6) * (tx1 + 2*tx2);  Fy2 = (Le/6) * (ty1 + 2*ty2)

    # Translate Gmsh tags → solver IDs
    o1 = mesh.gmsh_to_solver_node[n1g]
    o2 = mesh.gmsh_to_solver_node[n2g]

    # Accumulate (shared nodes get contributions from adjacent edges)
    nodal_forces.setdefault(o1, [0., 0.])
    nodal_forces.setdefault(o2, [0., 0.])
    nodal_forces[o1][0] += Fx1;  nodal_forces[o1][1] += Fy1
    nodal_forces[o2][0] += Fx2;  nodal_forces[o2][1] += Fy2

for nid, (fx, fy) in nodal_forces.items():
    ops.load(nid, fx, fy)

print(f"Pressure applied to {len(nodal_forces)} nodes")

## Part 4 — Analysis & Post-processing

Static linear analysis in one load step. After solving, we extract
displacements and stresses into plain numpy arrays and call `ops.wipe()`
to release OpenSees. From this point on, OpenSees is completely done.

In [ ]:
# --- Solve ---
#
# Static analysis setup:
#   Transformation constraints → handles multi-point constraints correctly
#   RCM numberer              → Reverse Cuthill-McKee bandwidth reduction
#   BandGeneral system        → banded solver (efficient for 2D meshes)
#   NormDispIncr test         → convergence on displacement increment norm
#   Newton algorithm          → full Newton-Raphson (1 iteration for linear)
#   LoadControl integrator    → apply full load in 1 step (λ = 1.0)

ops.constraints("Transformation")
ops.numberer("RCM")
ops.system("BandGeneral")
ops.test("NormDispIncr", 1.0e-8, 10)
ops.algorithm("Newton")
ops.integrator("LoadControl", 1.0)
ops.analysis("Static")

ok = ops.analyze(1)
print("CONVERGED" if ok == 0 else "FAILED")

In [ ]:
# --- Extract results into plain numpy arrays ---
#
# We pull everything out of OpenSees into numpy and then wipe.
# This keeps the solver window as small as possible.
#
# Results are indexed by raw Gmsh node position (via tag_to_idx) so they
# align with node_coords for plotting and Gmsh views.  We use
# solver_to_gmsh_node to translate solver IDs back to Gmsh tags.

nElem = connectivity.shape[0]
nNode = len(node_tags)

# -- Displacements --
disp = np.zeros((nNode, 2))
for solver_id, gmsh_tag in mesh.solver_to_gmsh_node.items():
    idx = tag_to_idx[gmsh_tag]
    disp[idx, 0] = ops.nodeDisp(solver_id, 1)   # ux
    disp[idx, 1] = ops.nodeDisp(solver_id, 2)   # uy

# -- Radial displacement --
# u_r = (x·ux + y·uy) / r  — projection onto radial direction.
# Guard against division by zero at the origin (orphan node).

r_nodes = np.sqrt(node_coords[:, 0]**2 + node_coords[:, 1]**2)
r_safe  = np.where(r_nodes > 1e-12, r_nodes, 1.0)
ur = (node_coords[:, 0]*disp[:, 0] + node_coords[:, 1]*disp[:, 1]) / r_safe

# -- Element stresses --
# tri31 (CST) returns one constant stress per element: [σxx, σyy, σxy]
# We iterate over solver element IDs and map back to Gmsh ordering.

sig_xx = np.zeros(nElem)
sig_yy = np.zeros(nElem)
sig_xy = np.zeros(nElem)

for solver_eid in range(1, mesh.n_elems + 1):
    # Map solver elem ID → Gmsh elem tag → position in elem_tags list
    gmsh_etag = mesh.solver_to_gmsh_elem[solver_eid]
    idx = elem_tags.index(gmsh_etag)
    s = ops.eleResponse(solver_eid, "stresses")
    sig_xx[idx] = s[0]
    sig_yy[idx] = s[1]
    sig_xy[idx] = s[2]

# -- Nodal-averaged σ_xx --
# CST gives piecewise-constant stress (jumps at element boundaries).
# Nodal averaging: for each node, average the stress from all elements
# sharing that node. This produces a smooth contour field.

conn_idx = np.array([[tag_to_idx[n] for n in row] for row in connectivity])
sig_xx_nodal = np.zeros(nNode)
node_count   = np.zeros(nNode)

for e in range(nElem):
    for ln in range(3):
        nidx = conn_idx[e, ln]
        sig_xx_nodal[nidx] += sig_xx[e]
        node_count[nidx]   += 1.0

node_count[node_count == 0] = 1.0
sig_xx_nodal /= node_count

print(f"u_r  range: [{ur.min():.6f}, {ur.max():.6f}] mm")
print(f"σ_xx range: [{sig_xx.min():.2f}, {sig_xx.max():.2f}] MPa")

# --- Done with OpenSees ---
ops.wipe()

## Part 5 — Matplotlib Plots

Quick visual checks using matplotlib before launching the Gmsh GUI.
These use the same numpy arrays — no solver or Gmsh calls needed.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

# Triangulation for matplotlib — uses array indices, not Gmsh tags
triang = mtri.Triangulation(node_coords[:, 0], node_coords[:, 1], conn_idx)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Radial displacement contour
ax = axes[0]
tripc = ax.tripcolor(triang, ur, shading='gouraud', cmap='coolwarm')
plt.colorbar(tripc, ax=ax, label=r"$u_r$ [mm]")
ax.set_title("Radial displacement")
ax.set_aspect('equal')

# (b) Nodal-averaged σ_xx contour
ax = axes[1]
tripc2 = ax.tripcolor(triang, sig_xx_nodal, shading='gouraud', cmap='RdYlBu_r')
plt.colorbar(tripc2, ax=ax, label=r"$\sigma_{xx}$ [MPa] (nodal avg)")
ax.set_title(r"$\sigma_{xx}$ — nodal averaged")
ax.set_aspect('equal')

# (c) Element-constant σ_xx (raw CST output)
ax = axes[2]
triplt = ax.tripcolor(triang, sig_xx, shading='flat', cmap='RdYlBu_r')
plt.colorbar(triplt, ax=ax, label=r"$\sigma_{xx}$ [MPa]")
ax.set_title(r"$\sigma_{xx}$ — element constant (CST)")
ax.set_aspect('equal')

# Overlay inner/outer boundaries
theta = np.linspace(0, np.pi/2, 100)
for ax in axes:
    ax.plot(inner_radius*np.cos(theta), inner_radius*np.sin(theta), "k--", lw=0.8)
    ax.plot(outer_radius*np.cos(theta), outer_radius*np.sin(theta), "k-", lw=0.8)
    ax.set_xlim(-10, outer_radius+15)
    ax.set_ylim(-10, outer_radius+15)

plt.tight_layout()
plt.show()

## Part 6 — Gmsh Post-processing Views

Pass the numpy result arrays back to pyGmsh for visualization in the Gmsh GUI.
The `g.view` methods are **solver-agnostic** — they accept plain arrays and
element/node tag lists, never importing OpenSees.

| Method                    | Data lives at… | Smoothing        |
|---------------------------|----------------|------------------|
| `add_element_scalar()`    | elements       | piecewise-const  |
| `add_element_vector()`    | elements       | piecewise-const  |
| `add_node_scalar()`       | nodes          | Gouraud-smooth   |
| `add_node_vector()`       | nodes          | Gouraud-smooth   |

In [ ]:
# --- Gmsh post-processing views ---
#
# Element-constant stress (raw CST):
g.view.add_element_scalar("sig_xx (elem)", elem_tags, sig_xx)
g.view.add_element_scalar("sig_yy (elem)", elem_tags, sig_yy)
g.view.add_element_scalar("sig_xy (elem)", elem_tags, sig_xy)

# Nodal-averaged stress (smooth):
ops_to_gmsh_tags = list(used_tags)  # only nodes connected to elements
g.view.add_node_scalar("sig_xx (nodal avg)", list(node_tags.astype(int)), sig_xx_nodal)

# Radial displacement (smooth):
g.view.add_node_scalar("u_r", list(node_tags.astype(int)), ur)

# Full displacement vector:
g.view.add_node_vector("displacement", list(node_tags.astype(int)), disp)

print(f"Created {g.view.count()} Gmsh views")

## Part 7 — Alternative paths: Numberer vs g2o vs manual

This notebook uses the **Numberer** path (recommended). Here's how all
three approaches compare:

| Scenario                           | Recommended path              |
|------------------------------------|-------------------------------|
| Standard workflow (most cases)     | `get_numbered_mesh()` + maps  |
| Quick prototype, standard elements | `g.g2o.transfer()`            |
| Custom edge loads (like this one!) | Numberer + raw `get_fem_data` |
| Bandwidth optimisation needed      | `get_numbered_mesh(method="rcm")` |
| Multi-material with different elems| Numberer + physical group maps|
| Teaching / understanding the flow  | This notebook!                |

The Numberer replaces the old manual `gmsh_to_ops` loop:

```python
# OLD (manual):
gmsh_to_ops = {}
new_id = 0
for gtag, coords in zip(node_tags, node_coords):
    if int(gtag) not in used_tags:
        continue
    new_id += 1
    gmsh_to_ops[int(gtag)] = new_id
    ops.node(new_id, float(coords[0]), float(coords[1]))

# NEW (Numberer):
mesh = g.mesh.get_numbered_mesh(dim=2, method="simple")
for i in range(mesh.n_nodes):
    ops.node(int(mesh.node_ids[i]), *mesh.node_coords[i])
# Maps: mesh.gmsh_to_solver_node, mesh.solver_to_gmsh_node
```

The `g.g2o.transfer()` wrapper (José Abell's gmsh2opensees) is even
simpler but gives less control over numbering and orphan filtering.

In [ ]:
# --- Check if gmsh2opensees is available ---
#
# g.g2o.is_available() returns True if the library is importable.
# If not, this cell just prints a message — nothing breaks.

print(f"gmsh2opensees wrapper: {g.g2o}")
print(f"Available: {g.g2o.is_available()}")

if g.g2o.is_available():
    # Example: one-liner transfer (commented out to avoid overwriting
    # the manual model we already built above).
    #
    # ops.wipe()
    # ops.model("basic", "-ndm", 2, "-ndf", 2)
    # ops.nDMaterial("ElasticIsotropic", 1, E, nu)
    # g.g2o.transfer(verbose=True)
    #
    # After transfer, you still apply BCs and loads manually:
    # for gtag in bottom_nodes.astype(int):
    #     ops.fix(int(gtag), 0, 1)   # note: gmsh2opensees uses Gmsh tags directly
    print("gmsh2opensees is installed — g.g2o.transfer() is ready to use.")
else:
    print("gmsh2opensees not installed. Using manual path (get_fem_data).")

## Part 8 — Launch Gmsh GUI & Finalize

In [ ]:
# Launch the Gmsh GUI to interactively explore the views
# (close the window to continue to the next cell)
g.launch_gui()

In [ ]:
# Release Gmsh resources
g.finalize()